# Individual Recommender Model

In this notebook, we build and test an individual movie recommender using the MovieLens 100K dataset.

The model predicts how users might rate movies they haven't seen before. Later, these predictions are used to create group recommendations and explain them using LIME and SHAP.

In [1]:
import pandas as pd
from pathlib import Path

## Loading the Data

Load the MovieLens ratings dataset. Each row has a user, a movie, the rating given by the user and a timestamp.

In [2]:
data_directory = Path("../data/ml-100k")

ratings = pd.read_csv(
    data_directory / "u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


## Training and testing splits

We use the predefined MovieLens 100K training and test sets. Train is used as training and test to check the predictions made by the model.
The predefined split also ensures that every user in the test set appears in the training set.

In [3]:
train = pd.read_csv(
    data_directory / "u1.base",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

test = pd.read_csv(
    data_directory / "u1.test",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

print("number of training ratings:", len(train))
print("number of test ratings:", len(test))

number of training ratings: 80000
number of test ratings: 20000


Sanity check

In [4]:
#check for users and movies that are in test but not in train
train_users = set(train["user_id"])
test_users = set(test["user_id"])

train_movies = set(train["movie_id"])
test_movies = set(test["movie_id"])

new_users = test_users - train_users
new_movies = test_movies - train_movies

print("Users in test but not train:", len(new_users))
print("Movies in test but not train:", len(new_movies))

Users in test but not train: 0
Movies in test but not train: 32


Checking how bad the cold start problem is.

In [ ]:
cs_movies = test["movie_id"].isin(new_movies)
cs_ratings = test[cs_movies]

print("number of cold start movies:", len(new_movies))
print("Test ratings involving cold start movies:", len(cs_ratings))
print("Percentage of test ratings:", len(cs_ratings) / len(test) * 100)

number of cold start movies: 32
Test ratings involving cold start movies: 32
Percentage of test ratings: 0.16


### Cold start check

All users in the test set are also in the in the training set. However, 32 movies in the test set are not in the training data.
Since the model has not learned movie-specific information for the 32 cold-start movies, we remove their ratings from the evaluation.
Which leaves 19968 test ratings for testing the models predictions.

In [ ]:

known_movies = test["movie_id"].isin(train_movies)
test_known = test[known_movies].copy()

print("Original test ratings:", len(test))
print("Evaluation ratings:", len(test_known))

Original test ratings: 20000
Evaluation ratings: 19968


## Matrix Factorization Recommender
(SVD ended up not being used)
We use an SVD model to learn users movie preferences from the training data.
The model then uses what it has learned to predict how users would rate other movies.

In [7]:
from surprise import Dataset, Reader, SVD
from surprise import accuracy

In [8]:
import sys
print(sys.executable)

c:\Users\antre\Desktop\Thesis\.venv\Scripts\python.exe


## Train the individual recommender

We use the SVD algorithm from the Surprise library to train the recommender on the MovieLens training data.
First, the ratings are prepared in the format required by Surprise and then we train the model to predict what a user would rate a movie

In [9]:
from surprise import Dataset, Reader, SVD
from surprise import accuracy

#tell Surprise that ratings range from 1 to 5
reader = Reader(rating_scale=(1, 5))

#convert pandas training data into Surprise's format (df is a dataframe with columns user_id, movie_id, rating)
train_data = Dataset.load_from_df(
    train[["user_id", "movie_id", "rating"]],
    reader
)

#convert it into the structure Surprise uses for training
trainset = train_data.build_full_trainset()

#create the SVD recommender
model = SVD(random_state=42)

#train the model
model.fit(trainset)

print("Model training successful.")

Model training successful.


## Evaluate the Individual Recommender

Test the model on ratings not included in the training
Compare the predicted ratings with the actual ratings using RMSE. Lower RMSE = better prediction

In [10]:
# Select the columns needed for testing
test_rtngs = test_known[["user_id", "movie_id", "rating"]]

# Convert the ratings into the format Surprise needs
testset = list(test_rtngs.itertuples(index=False, name=None))

# Make predictions
predictions = model.test(testset)

#Calculate the error
rmse = accuracy.rmse(predictions)

RMSE: 0.9557


## Inspect individual predictions

Inspect several test-set predictions to compare the actual user ratings with the ratings predicted by the trained SVD model.
Suprise returns a list of prediction objects, each prediction contains information about one user-movie ratings (userid,movieid,rating,predicted rating)

In [11]:
# Look at the first 10 predictions
# uid  user id, iid item id, r_ui actual rating, est predicted rating, these are attributes given by Suprise
for prediction in predictions[:10]:
    print("\nuser:", prediction.uid)
    print("movie:", prediction.iid)
    print("actual rating:", prediction.r_ui)
    print("predicted rating:", round(prediction.est, 2))


user: 1
movie: 6
actual rating: 5
predicted rating: 3.47

user: 1
movie: 10
actual rating: 3
predicted rating: 4.24

user: 1
movie: 12
actual rating: 5
predicted rating: 4.76

user: 1
movie: 14
actual rating: 5
predicted rating: 4.25

user: 1
movie: 17
actual rating: 3
predicted rating: 3.43

user: 1
movie: 20
actual rating: 4
predicted rating: 3.45

user: 1
movie: 23
actual rating: 4
predicted rating: 4.41

user: 1
movie: 24
actual rating: 3
predicted rating: 3.3

user: 1
movie: 27
actual rating: 2
predicted rating: 3.36

user: 1
movie: 31
actual rating: 3
predicted rating: 3.66


## Group Construction

Groups of 4 users are created to compare how different group types affect the recommendations and their explanations.
We will use random, similar and dissimilar groups.
To measure how similar users are, we use their ratings from the training set only.

In [12]:
# Create a user-movie matrix from the training data
# Pivot table is a panda function that reorganizes the data into a table (matrix) where rows are users, columns are movies, and values are ratings. 
# It is useful for collaborative filtering because it allows us to see which users have rated which movies and what those ratings are.
user_movie_matrix = train.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

user_movie_matrix.shape

(943, 1650)

Visualization of the table

In [13]:
user_movie_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,NaN,4.0,1.0,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
import numpy as np

#notna() is a pandas function that checks if the value is not Nan which means the user rated the movie.
#astype(int) converts the boolean values to integers (1 for true, 0 for false)
#has_rated is true if user rated, false if user did not.
has_rated = user_movie_matrix.notna()

# Number of movies each pair of users has rated in common
#
bool_has_rated = has_rated.astype(int)

#@ means matrix multiplication, .T means transpose, we multiply them to get the number of movies each pair of users has rated in common
common_movies = bool_has_rated @ bool_has_rated.T

common_movies.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,135,6,3,2,21,33,32,5,2,22,...,37,7,25,10,18,7,25,7,14,35
2,6,40,3,2,1,13,8,1,3,5,...,7,11,23,15,18,8,12,5,7,7
3,3,3,28,3,1,6,2,3,0,3,...,3,1,9,4,5,2,9,2,4,1
4,2,2,3,14,1,0,3,2,0,1,...,1,1,6,3,4,1,5,3,5,2
5,21,1,1,1,91,16,29,6,2,9,...,34,4,9,2,12,4,17,4,12,33


### Checking shared movie counts

Check how many movies each pair of users has rated in common.
Only count each pair once and exclude users compared with themselves.
This helps us decide how many shared movies to require before calculating similarity.

In [15]:
#keep only unique user pairs (upper triangle)
#excluding users compared with themselves
#np.ones creates a matrix filled with 1s and dtype makes that matrix boolean
#common_movies.shape gives the dimensions of the overlap matrix. 
all_pairs = np.ones(common_movies.shape, dtype=bool)

#k=1 tells python to start one diagonal up from the diagonal(users compared with themselves)
#np.triu creates an upper triangular matrix from the boolean matrix
upper_triangle = np.triu(all_pairs, k=1)

#.value gets the values as a numpy arrray 
#[upper_triangle] is a boolean mask that selects only the upper triangle of the overlap matrix.
pair_overlaps = common_movies.values[upper_triangle]

#pd.Series is a pandas function that creates a one dimension array.
#.describe() is a pandas function that gives summary statistics of the data.
pd.Series(pair_overlaps).describe()

count    444153.000000
mean         12.127985
std          17.822309
min           0.000000
25%           2.000000
50%           6.000000
75%          14.000000
max         332.000000
dtype: float64

### Choosing the minimum number of shared movies

Check how many user pairs have at least 5, 10, 15, 20, or 30 movies in common.

Decided to use a minimum of 10 shared movies when calculating Pearson similarity. This gives enough common ratings to compare users while still keeping a reasonable number of user pairs.

In [16]:
for threshold in [5, 10, 15, 20, 30]:
    valid_pairs = (pair_overlaps >= threshold).sum()
    percentage = valid_pairs / len(pair_overlaps) * 100

    print(
        f"At least {threshold} co-rated movies: "
        f"{valid_pairs} pairs ({percentage:.2f}%)"
    )

At least 5 co-rated movies: 260627 pairs (58.68%)
At least 10 co-rated movies: 157659 pairs (35.50%)
At least 15 co-rated movies: 106315 pairs (23.94%)
At least 20 co-rated movies: 77128 pairs (17.37%)
At least 30 co-rated movies: 46038 pairs (10.37%)


### Calculating user similarity

We are using Pearson correlation to compare users based on their movie ratings.
We are only calculating similarity when two users have rated at least 10 movies in common.
A positive correlation means users have similar rating patterns unlike a negative correlation which means they tend to rate movies differently.
The result is a matrix showing the similarity between each pair of users.

In [17]:
# transposes the matrix so that users are columns and movies are rows, which is required for the correlation calculation

rtngs_by_usr = user_movie_matrix.T

# Calculate Pearson similarity between users
# min periods=10 means that we only calculate the correlation if there are at least 10 movies that both users have rated.
user_pearsonsim = rtngs_by_usr.corr(
    method="pearson",
    min_periods=10
)

user_pearsonsim.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,NaN,NaN,NaN,0.147833,0.298108,0.234765,NaN,NaN,-0.080917,...,0.008439,NaN,0.203564,-0.235457,0.551592,NaN,0.210060,NaN,-0.193489,0.243038
2,NaN,1.0,NaN,NaN,NaN,0.484802,NaN,NaN,NaN,NaN,...,NaN,-0.516398,0.293674,0.618042,0.170254,NaN,-0.129460,NaN,NaN,NaN
3,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,0.147833,NaN,NaN,NaN,1.000000,0.025565,0.237864,NaN,NaN,NaN,...,0.293481,NaN,NaN,NaN,0.568360,NaN,0.039459,NaN,0.233984,0.396352


### Creating candidate groups

randomly select groups of 4 users and calculate the average Pearson similarity between the members.
Each group has 6 possible pairs of users. only keep groups where all 6 similarities can be calculated.
repeat this until we have 10,000 candidate groups.

In [18]:
import random
from itertools import combinations

random.seed(42)
group_size = 4
num_of_groups = 10000
users = list(user_pearsonsim.index)
group_pool = []

while len(group_pool) < num_of_groups:

    # Choose 4 different users randomly
    group = random.sample(users, group_size)
    
    # Find the 6 pairs of users
    pairs = list(combinations(group, 2))

    # Get the similarity for each pair
    similarities = []
    #user_pearsonsim.loc[user1, user2] gets the similarity score between user1 and user2 from the user_pearsonsim dataframe.
    # append add an item to the end of a list. In this case, we are adding the similarity score for each pair of users to the similarities list.
    for user1, user2 in pairs:
        similarity = user_pearsonsim.loc[user1, user2]
        similarities.append(similarity)

    # Only keep groups with valid similarities
    if not any(pd.isna(similarities)):

        avg_similarity = np.mean(similarities)

        group_pool.append({
            "users": group,
            "avg_similarity": avg_similarity
        })

group_pool = pd.DataFrame(group_pool)
group_pool.head()

,users,avg_similarity
0,"[666, 719, 559, 430]",0.364220
1,"[522, 506, 94, 774]",0.194374
2,"[553, 774, 748, 707]",0.015219
3,"[479, 885, 747, 56]",0.191727
4,"[454, 828, 883, 877]",0.156716


### Group similarity distribution

Check the average similarity scores of the 10,000 candidate groups.
use the bottom 25% of groups as lower similarity groups and the top 25% as higher similarity groups.

In [19]:
group_pool["avg_similarity"].describe()

count    10000.000000
mean         0.159906
std          0.112049
min         -0.228449
25%          0.083022
50%          0.161079
75%          0.234232
max          0.587238
Name: avg_similarity, dtype: float64

Based on the candidate groups, most groups have a positive average similarity.

Instead of requiring dissimilar groups to have negative similarity, the groups are divided using the observed similarity distribution:

Groups in the bottom 25% are considered dissimilar groups.
Groups in the top 25% are considered similar groups.

This creates two clearly different group types without requiring fixed similarity values to be chosen manually.

In [20]:
#quantile gives the value below which a given percentage of observations in a group of observation fall.
low_thresh = group_pool["avg_similarity"].quantile(0.25)
high_thresh = group_pool["avg_similarity"].quantile(0.75)

#save similar and dissimilar. 
dissimilar_groups = group_pool[
    group_pool["avg_similarity"] <= low_thresh
].copy()

similar_groups = group_pool[
    group_pool["avg_similarity"] >= high_thresh
].copy()

print("Lower threshold:", round(low_thresh, 3))
print("High threshold:", round(high_thresh, 3))
print("possible dissimilar groups:", len(dissimilar_groups))
print("possible similar groups:", len(similar_groups))

Lower threshold: 0.083
High threshold: 0.234
possible dissimilar groups: 2500
possible similar groups: 2500


### Selecting groups without repeating users

We create a function that we can reuse for both group types.
We select groups from the group pool while avoiding repeated users.
We first shuffle the candidate groups. Then go through them one by one and only keep a group if none of its users has already been selected.
This helps us create groups with different members and avoids using the same users too many times.

In [21]:
#this function takes a dataframe of groups and returns a new dataframe containing only the groups that do not have any overlapping users.
def get_groups(group_pool):

    chosen_groups = []
    already_used = set()

    # Shuffle the groups
    # frac=1 means we want to return all rows, but in a random order.
    # random_state=42 same as seed 42
    shuffled_groups = group_pool.sample(
        frac=1,
        random_state=42
    )

    for index, row in shuffled_groups.iterrows():

        group = row["users"]

        # Check if any user was already selected
        is_repeated = False

        for user in group:
            if user in already_used:
                is_repeated = True

        # Keep the group if no users are repeated
        # update() adds the users in the current group to the set of used users, so that we can check for overlaps in future iterations.
        if is_repeated == False:
            chosen_groups.append(row)
            already_used.update(group)
    #Convert the list of selected groups into a pandas DataFrame and return it. Makes it easier to work with later on.
    return pd.DataFrame(chosen_groups)

In [22]:
chosen_similar = get_groups(similar_groups)
chosen_dissimilar = get_groups(dissimilar_groups)

print("similar groups with no overlap:", len(chosen_similar))
print("dissimilar groups with no overlap:", len(chosen_dissimilar))

similar groups with no overlap: 89
dissimilar groups with no overlap: 94


89 similar groups and 94 dissimilar groups, without repeating users within each group type.
For the experiment, we will select the same number of groups from each type to make the comparison fair. (later chosen 80 groups of each type)

### Random Groups

We also create random groups to compare with the similar and dissimilar groups.
Groups have 4 random users. We make sure that a user does not appear in more than one random group.
Users can still appear in groups of different types.

In [23]:
# Create random groups
random.seed(42)
#make a seperate list of users to not change the original.
available_users = users.copy()
#randomly rearrange the list of users
random.shuffle(available_users)

random_grps = []
# should be 943/4 = 235 groups with 3 users left over
for i in range(0, len(available_users) - 3, 4):
    #select a group of 4 users from the shuffled list
    group = available_users[i:i+4]
    random_grps.append(group)

print("Random groups:", len(random_grps))
print("First random group:", random_grps[0])

Random groups: 235
First random group: [385, 893, 399, 653]


### Final Experimental Groups

Since the groups are 89, 94, 235 we will only use 80 from each group to make the comparison more balanced
Each group has 4 users. Users are not repeated within the same group type but they can appear in different group types.

In [24]:
# Use the same number of groups for each type
num_groups = 80
#We use the sample() function to randomly select num_groups from each DataFrame
final_similar = chosen_similar.sample(n=num_groups, random_state=42)
final_dissimilar = chosen_dissimilar.sample(n=num_groups, random_state=42)
#:num_groups is the number of groups we want to select from each type (similar, dissimilar, random). 
final_random = random_grps[:num_groups]

print("Number of similar groups:", len(final_similar))
print("Number of dissimilar groups:", len(final_dissimilar))
print("Number of random groups:", len(final_random))

Number of similar groups: 80
Number of dissimilar groups: 80
Number of random groups: 80


### Checking group similarity

Just a check to be sure similarity of groups is where it is supposed to be (similar groups have higher similarity dissimilar lower and random in between)

In [25]:
# Calculate the average similarity of each random group
random_similarities = []

for group in final_random:
    pairs = list(combinations(group, 2))

    similarities = []
#the for loop iterates over each pair of users in the group. For each pair, it retrieves the similarity score from the user_pearsonsim DataFrame and appends it to the similarities list.
    for user_a, user_b in pairs:
        #user_pearsonsim.loc[user_a, user_b] gets the similarity score between user_a and user_b from the user_pearsonsim dataframe.
        similarity = user_pearsonsim.loc[user_a, user_b]
        similarities.append(similarity)

    # Ignore missing similarity values
    #np.nanmean calculates the mean of an array while ignoring NaN values. 
    #This is important because some user pairs may not have a similarity score, and we don't want those missing values to affect the average.
    average = np.nanmean(similarities)
    random_similarities.append(average)

print("Average similarity of similar groups:", final_similar["avg_similarity"].mean())
print("Average similarity of dissimilar groups:", final_dissimilar["avg_similarity"].mean())
print("Average similarity of random groups:", np.nanmean(random_similarities))

Average similarity of similar groups: 0.2939312404220689
Average similarity of dissimilar groups: 0.018455094828218423
Average similarity of random groups: 0.19352135867013626


C:\Users\antre\AppData\Local\Temp\ipykernel_3804\2143804911.py:17: RuntimeWarning: Mean of empty slice
  average = np.nanmean(similarities)


### Checking missing similarities

Some random groups may not have enough shared movie ratings to calculate Pearson similarity.
Checking how many random groups have no valid similarity score.

In [26]:
# Count random groups with no valid similarity score
missing_groups = pd.isna(random_similarities).sum()

print("Random groups:", len(final_random))
print("Groups with no valid similarity:", missing_groups)
print("Groups with valid similarity:", len(final_random) - missing_groups)

Random groups: 80
Groups with no valid similarity: 22
Groups with valid similarity: 58


### Random group similarity

Out of 80 random groups 58 of them have a valid average similarity score and 22 do not.
This happens because some users have fewer than 10 movies rated in common.
Removing groups based on similarity availability could affect the random selection.
The average similarity of random groups is calculated only from groups with available similarity scores.

## Group recommendations

Generate group recommendations by combining individual SVD predictions using aggregation strategies.
Single group first.

In [27]:
# Predict ratings for a movie for a group of users
#iloc is a pandas function that allows us to select rows and columns by their integer position.
group = final_similar.iloc[0]["users"]
movie_id = 1
predicted_ratings = []

for user_id in group:
    prediction = model.predict(user_id, movie_id)
    predicted_ratings.append(prediction.est)

print("group members:", group)
print("movie:", movie_id)
print("predicted ratings:", predicted_ratings)

group members: [556, 342, 881, 373]
movie: 1
predicted ratings: [np.float64(4.461424262155591), np.float64(3.876452710984175), np.float64(3.8524246143407024), np.float64(3.6672213986422992)]


In [28]:
# Calculate group scores using two aggregation strategies

avg_score = np.mean(predicted_ratings)
lm_score = np.min(predicted_ratings)

print("avg group score:", round(avg_score, 2))
print("least misery group score:", round(lm_score, 2))

avg group score: 3.96
least misery group score: 3.67


Finding movies that none of the 4 users have rated. 

That is to recommend everyone something new.


In [29]:
# Find movies alrteady rated by anyone
#train[train["user_id"].isin(group)] filters the training data to include only the ratings made by users in the group.
group_ratings = train[train["user_id"].isin(group)]

already_rated_movies = set(group_ratings["movie_id"])

# Find all movies available in our training data
all_movies = set(train["movie_id"])

# Keep movies that no group member has rated
available_movies = sorted(all_movies - already_rated_movies)

print("movies available in training:", len(all_movies))
print("movies rated by group members:", len(already_rated_movies))
print("movies available for recommendation:", len(available_movies))

movies available in training: 1650
movies rated by group members: 425
movies available for recommendation: 1225


In [30]:
# Calculate average and least misery for each candidate movie
group_scores = []
#Outer is choosing a movie from the canditates 
for movie_id in available_movies:

    #predict how each group member would rate this movie
    predicted_ratings = []
    #inner loop is going through each user in the group and predicting how they would rate the current movie.
    for user_id in group:
        prediction = model.predict(user_id, movie_id)
        #prediction.est is the estimated rating for the user and movie. We append this estimated rating to the predicted_ratings list for each user in the group.
        predicted_ratings.append(prediction.est)

    # Combine the four predictions into group scores
    average = np.mean(predicted_ratings)
    least_misery = np.min(predicted_ratings)

    # Save the scores for this movie
    group_scores.append({
        "movie_id": movie_id,
        "avg_score": average,
        "lm_score": least_misery
    })

group_scores = pd.DataFrame(group_scores)

group_scores.head()

,movie_id,avg_score,lm_score
0,2,3.272267,2.822425
1,5,3.315002,3.131636
2,6,3.549533,3.464173
3,10,3.872525,3.112903
4,16,3.311339,2.690938


In [31]:
# Top 10 movies using Average aggregation
top10_avg = group_scores.sort_values(
    by="avg_score", ascending=False
).head(10)

# Top 10 movies using Least Misery aggregation
top10_lm = group_scores.sort_values(
    by="lm_score", ascending=False
).head(10)

print("Top 10 average:")
display(top10_avg)

print("Top 10 least misery:")
display(top10_lm)

Top 10 average:


,movie_id,avg_score,lm_score
125,313,4.430533,4.208358
99,272,4.277629,4.065548
325,657,4.181085,3.915725
107,285,4.156172,3.654059
328,661,4.137023,3.796913
293,611,4.136663,3.938911
320,647,4.130093,3.891307
20,45,4.123724,3.825326
726,1142,4.111374,3.851280
327,659,4.109826,3.859505


Top 10 least misery:


,movie_id,avg_score,lm_score
125,313,4.430533,4.208358
99,272,4.277629,4.065548
293,611,4.136663,3.938911
325,657,4.181085,3.915725
320,647,4.130093,3.891307
327,659,4.109826,3.859505
726,1142,4.111374,3.851280
237,522,4.002638,3.831454
20,45,4.123724,3.825326
319,646,3.981316,3.824739


In [32]:
# Compare movies recommended by both strategies
average_movies = set(top10_avg["movie_id"])
least_misery_movies = set(top10_lm["movie_id"])

common_movies = average_movies & least_misery_movies

print("Movies recommended by both:", len(common_movies))
print("Common movie ids:", sorted(common_movies))

Movies recommended by both: 8
Common movie ids: [45, 272, 313, 611, 647, 657, 659, 1142]


### Group Recommendation Function

Create a reusable function to calculate average and least misery scores for a group of users.

Only movies that none of the group members rated in the training data are considered.

In [33]:
def group_recommendation(group):

    # Find movies that the group has already rated
    #train[train["user_id"].isin(group)] filters the training data to include only the ratings made by users in the group.
    group_ratings = train[train["user_id"].isin(group)]
    already_rated_movies = set(group_ratings["movie_id"])
    available_movies = sorted(all_movies - already_rated_movies)
    group_scores = []

    # Calculate scores for each candidate movie
    for movie_id in available_movies:
        predicted_ratings = []

        for user_id in group:
            prediction = model.predict(user_id, movie_id)
            predicted_ratings.append(prediction.est)

        average = np.mean(predicted_ratings)
        least_misery = np.min(predicted_ratings)

        group_scores.append({
            "movie_id": movie_id,
            "avg_score": average,
            "lm_score": least_misery
        })
    
    return pd.DataFrame(group_scores)

In [34]:
# Test the function with a group of users
#final_similar.iloc[0]["users"] selects the first row of the final_similar DataFrame 
# and retrieves the value in the "users" column, which is a list of user IDs representing a group of similar users.
test_grp = final_similar.iloc[0]["users"]

test_scrs = group_recommendation(test_grp)

print("Number of candidate movies:", len(test_scrs))
display(test_scrs.head())

Number of candidate movies: 1225


,movie_id,avg_score,lm_score
0,2,3.272267,2.822425
1,5,3.315002,3.131636
2,6,3.549533,3.464173
3,10,3.872525,3.112903
4,16,3.311339,2.690938


In [35]:
# Convert random groups into a DataFrame
random_df = pd.DataFrame({
    "users": final_random,
    "avg_similarity": random_similarities
})

display(random_df.head())

,users,avg_similarity
0,"[385, 893, 399, 653]",0.146103
1,"[704, 754, 50, 928]",NaN
2,"[845, 482, 54, 731]",NaN
3,"[457, 222, 402, 587]",0.343456
4,"[303, 756, 790, 488]",0.204575


In [36]:
# Add a label to each group category
#[group_type] is a new column in the DataFrame that indicates the type of group
final_similar["group_type"] = "similar"
final_dissimilar["group_type"] = "dissimilar"
random_df["group_type"] = "random"

# Combine all groups into one DataFrame
grp_df = pd.concat(
    [final_similar, final_dissimilar, random_df],
    #ignore_index=True resets the index of the new DataFrame to be a continuous range of integers.
    ignore_index=True
)

print("Total groups:", len(grp_df))
display(grp_df.head())

Total groups: 240


,users,avg_similarity,group_type
0,"[556, 342, 881, 373]",0.249534,similar
1,"[296, 265, 869, 721]",0.300932,similar
2,"[650, 690, 145, 246]",0.338521,similar
3,"[763, 64, 6, 363]",0.324129,similar
4,"[251, 831, 327, 901]",0.256486,similar


In [37]:
# Check that every group contains exactly 4 different users

for group_type in grp_df["group_type"].unique():

    groups = grp_df[grp_df["group_type"] == group_type]

    valid_groups = 0

    for group in groups["users"]:
        if len(group) == 4 and len(set(group)) == 4:
            valid_groups += 1

    print(group_type, "valid groups:", valid_groups, "out of", len(groups))

similar valid groups: 80 out of 80
dissimilar valid groups: 80 out of 80
random valid groups: 80 out of 80


In [38]:
# Check that users do not repeat within the same category
#the loop iterates over each unique group type (similar, dissimilar, random) in the grp_df DataFrame. 
for group_type in grp_df["group_type"].unique():
    # For each group type, it filters the DataFrame to get only the groups of that type. 
    groups = grp_df[grp_df["group_type"] == group_type]
# It then initializes an empty list already_used to keep track of all users that have been seen in the current group type.
# extend() is a list method that adds all elements of an iterable (in this case, the users in the group) to the end of the list.
    already_used = []
# For each group in the filtered DataFrame, it extends the already_used list with the users in that group. 
    for group in groups["users"]:
        already_used.extend(group)
#it calculates the total number of user appearances (total_users) and the number of unique users (unique_users) by converting already_used to a set.
    total_users = len(already_used)
    unique_users = len(set(already_used))
# Finally, it prints out the group type, total user appearances, unique users, and repeated appearances (total_users - unique_users).
    print(group_type)
    print("Total user appearances:", total_users)
    print("Unique users:", unique_users)
    print("Repeated:", total_users - unique_users)
    

similar
Total user appearances: 320
Unique users: 320
Repeated: 0
dissimilar
Total user appearances: 320
Unique users: 320
Repeated: 0
random
Total user appearances: 320
Unique users: 320
Repeated: 0


In [39]:

#give each group an id.
grp_df["group_id"] = range(1, 241)

# Save all groups to a CSV file
grp_df.to_csv("../results/groups.csv", index=False)

print("saved!")

saved!


### Generating recommendations for all groups

We use our group recommendation function to generate movie scores for each group, using Average and Least Misery aggregation.

In [40]:
# Test the recommendation function on one group from each category
# 0, 80, and 160 are indices of the first group in each category (similar, dissimilar, random) in the grp_df DataFrame.
for i in [0, 80, 160]:
    #group = grp_df.iloc[i]["users"] retrieves the list of user IDs for the group at index i in the grp_df DataFrame.
    group = grp_df.iloc[i]["users"]
    #group_type = grp_df.iloc[i]["group_type"] retrieves the type of group (similar, dissimilar, random) for the group at index i in the grp_df DataFrame.
    group_type = grp_df.iloc[i]["group_type"]
    scores = group_recommendation(group)

    print("type of group:", group_type)
    print("possible movies:", len(scores))
    print()

type of group: similar
possible movies: 1225

type of group: dissimilar
possible movies: 1187

type of group: random
possible movies: 1101



### Generating Recommendations for All 240 Groups

We use the trained SVD model to generate individual rating predictions and aggregate them using Average and Least Misery for every group.

In [41]:
# Generate recommendations for all 240 groups

#grp_recommends is a list that will store the recommendations for each group. 
grp_recommends = []
#iterrows goes though the dataframe one row at a time 
# we use _ as a placeholder for the index because we don't need it in this case. We only care about the row data.
for _, row in grp_df.iterrows():

    group_id = row["group_id"]
    group = row["users"]
    scores = group_recommendation(group)

    # Save the top 10 movies for each strategy
    # nlargest() is a pandas function that returns the n largest values from a DataFrame or Series. 
    # In this case, it is used to get the top 10 movies based on average score and least misery score.
    top_avg = scores.nlargest(10, "avg_score")
    top_lm = scores.nlargest(10, "lm_score")
     
    #Each recommendation is a dictionary containing the group ID, the top 10 movies based on average score, and the top 10 movies based on least misery score.
    grp_recommends.append({
        "group_id": group_id,
        "average": top_avg,
        "least_misery": top_lm
    })

print("Groups processed:", len(grp_recommends))

Groups processed: 240


In [42]:
# Get the recommendations for the first group
first_grp = grp_recommends[0]

print("group id:", first_grp["group_id"])
print("Top 10  average:")
display(first_grp["average"])
print("Top 10 least misery:")
display(first_grp["least_misery"])

group id: 1
Top 10  average:


,movie_id,avg_score,lm_score
125,313,4.430533,4.208358
99,272,4.277629,4.065548
325,657,4.181085,3.915725
107,285,4.156172,3.654059
328,661,4.137023,3.796913
293,611,4.136663,3.938911
320,647,4.130093,3.891307
20,45,4.123724,3.825326
726,1142,4.111374,3.851280
327,659,4.109826,3.859505


Top 10 least misery:


,movie_id,avg_score,lm_score
125,313,4.430533,4.208358
99,272,4.277629,4.065548
293,611,4.136663,3.938911
325,657,4.181085,3.915725
320,647,4.130093,3.891307
327,659,4.109826,3.859505
726,1142,4.111374,3.851280
237,522,4.002638,3.831454
20,45,4.123724,3.825326
319,646,3.981316,3.824739


In [43]:
# Combine all recommendations into one table
rec_table = []

for result in grp_recommends:

    group_id = result["group_id"]
    
    for strategy in ["average", "least_misery"]:
        #make a copy of the top movies for the current strategy to avoid modifying the original DataFrame.
        top_movies = result[strategy].copy()
        
        top_movies["group_id"] = group_id
        top_movies["strategy"] = strategy
            
        rec_table.append(top_movies)
# Concatenate all recommendation tables into a single DataFrame
recs_df = pd.concat(rec_table, ignore_index=True)

print("Total recommendation rows:", len(recs_df))
display(recs_df.head())

Total recommendation rows: 4800


,movie_id,avg_score,lm_score,group_id,strategy
0,313,4.430533,4.208358,1,average
1,272,4.277629,4.065548,1,average
2,657,4.181085,3.915725,1,average
3,285,4.156172,3.654059,1,average
4,661,4.137023,3.796913,1,average


In [44]:
# Save recommendations to a CSV file
recs_df.to_csv("../results/recommendations.csv", index=False)

print("Recommendations saved successfully!")

Recommendations saved successfully!


In [45]:
# Compare Top10 recommendations for all groups
overlaps = []

for result in grp_recommends:

    average_movies = set(result["average"]["movie_id"])
    misery_movies = set(result["least_misery"]["movie_id"])
    common_movies = average_movies & misery_movies
    overlaps.append(len(common_movies))

print("Average common movies:", np.mean(overlaps))
print("Minimum common movies:", min(overlaps))
print("Maximum common movies:", max(overlaps))

Average common movies: 6.533333333333333
Minimum common movies: 3
Maximum common movies: 10


In [46]:
# Compare recommendation overlap by group type
#all
grp_df["overlap"] = overlaps

print(grp_df.groupby("group_type")["overlap"].mean())

group_type
dissimilar    6.300
random        6.875
similar       6.425
Name: overlap, dtype: float64


In [47]:
grp_df.to_csv("../results/groups.csv", index=False)

## Explainability experiment SVD perturbations

In [48]:

# Choose one user and one movie to investigate
user_id = 556
movie_id = 313

# Get the original SVD prediction
og_prediction = model.predict(user_id, movie_id).est

# Find the user's ratings in the training data
user_history = train[train["user_id"] == user_id]

print("Og prediction:", og_prediction)
print("Number of historical ratings:", len(user_history))


Og prediction: 4.559519027993886
Number of historical ratings: 44


In [49]:

# Select the first historical rating from this user
remove_rtng = user_history.iloc[0]

# Create a modified training dataset without that rating
perturbed_train = train.drop(index=remove_rtng.name)

print("removed:")
print(remove_rtng)
print("\nOriginal training rows:", len(train))
print("Modified training rows:", len(perturbed_train))


removed:
user_id            556
movie_id            12
rating               5
timestamp    882136440
Name: 42702, dtype: int64

Original training rows: 80000
Modified training rows: 79999


In [50]:

# Convert the modified ratings into Surprise's format
perturbed_data = Dataset.load_from_df(
    perturbed_train[["user_id", "movie_id", "rating"]],
    reader
)

# Build a training set from the modified data
perturbed_trainset = perturbed_data.build_full_trainset()

# Train a NEW SVD model with the same settings
perturbed_model = SVD(random_state=42)
perturbed_model.fit(perturbed_trainset)

# Predict the same movie for the same user
new_prediction = perturbed_model.predict(user_id, movie_id).est

# Compare both predictions
diff = new_prediction - og_prediction

print("Original prediction:", og_prediction)
print("New prediction:", new_prediction)
print("diff:", diff)


Original prediction: 4.559519027993886
New prediction: 4.547244770900173
diff: -0.012274257093713103


In [51]:

# Select a rating from another user and another movie
other_rating = train[
    (train["user_id"] != user_id) &
    (train["movie_id"] != movie_id)
].iloc[0]

# Remove that rating from a new copy
control_train = train.drop(index=other_rating.name)

# Prepare the control training data
control_data = Dataset.load_from_df(
    control_train[["user_id", "movie_id", "rating"]],
    reader
)

# Train a separate control model
control_model = SVD(random_state=42)
control_model.fit(control_data.build_full_trainset())

# Predict the original user and movie again
control_prediction = control_model.predict(user_id, movie_id).est

print("Og prediction:", og_prediction)
print("rating removed:", new_prediction)
print("Other user rating removed:", control_prediction)
print("Control diff:", control_prediction - og_prediction)


Og prediction: 4.559519027993886
rating removed: 4.547244770900173
Other user rating removed: 4.5762658254779005
Control diff: 0.01674679748401431


In [52]:

# Shuffle the original training data without removing anything
#sample(frac=1) randomly rearranges the rows of the DataFrame. frac=1 means we want to return all rows, but in a random order.
shuffled_train = train.sample(frac=1, random_state=123)

# Convert the shuffled data to Surprise format
shuffled_data = Dataset.load_from_df(
    shuffled_train[["user_id", "movie_id", "rating"]],
    reader
)

# Train SVD with the same settings
shuffled_model = SVD(random_state=42)
shuffled_model.fit(shuffled_data.build_full_trainset())

# Compare predictions
shuffled_prediction = shuffled_model.predict(user_id, movie_id).est

print("Og prediction:", og_prediction)
print("Shuffled prediction:", shuffled_prediction)
print("diff:", shuffled_prediction - og_prediction)


Og prediction: 4.559519027993886
Shuffled prediction: 4.207481171365073
diff: -0.3520378566288134


## Initial perturbation experiments

We investigated whether removing a historical rating and retraining Surprise SVD could provide a meaningful explanation of individual predictions.
For User 556 and Movie 313:
Original prediction: 4.5595
Removing one of the user's ratings: 4.5472
Removing another user's rating: 4.5763
Shuffling the training rows without removing ratings: 4.2075
The large change after shuffling shows that training order can substantially affect predictions. So, retraining the model cant help us explain the differencess.
